# Python `lambda`, `map`, `filter`, `reduce` 
---

| Function | Does | Returns |
|---|---|---|
| `lambda` | defines a throwaway, one-line function | a function object |
| `map(fn, iterable)` | applies `fn` to every item | lazy iterator (1-to-1 transform) |
| `filter(fn, iterable)` | keeps items where `fn` is `True` | lazy iterator (subset) |
| `reduce(fn, iterable)` | folds all items into a single value | one value (n-to-1 collapse) |

**Mental model:** `map` **transforms**. `filter` **selects**. `reduce` **collapses**.

---

### `lambda` — anonymous inline function

**Syntax:** `lambda arguments: expression` — always returns the expression. One expression only, no statements, no multi-line bodies.

```python
# Normal function
def square(x):
    return x * 2

# Equivalent lambda
square = lambda x: x * 2

# No args
lambda: 42

# Multiple args
lambda x, y: x + y

# Ternary inside a lambda
classify = lambda x: "even" if x % 2 == 0 else "odd"
```

##### Most common real use — inline with `key=`
```python
people = [{'name': 'Bob', 'age': 30}, {'name': 'Ana', 'age': 25}]

sorted(people, key=lambda p: p['age'])       # sort by age
min(people, key=lambda p: p['age'])           # youngest person
sorted(people, key=lambda p: (p['age'], p['name']))  # multi-key sort
```

##### Pitfall
Don't assign a lambda to a variable and use it like a named function — use `def` instead. Lambda is meant for throwaway, inline use (`sorted`, `map`, `filter`, `key=`), not for defining reusable functions.

```python
# Bad style (works, but not idiomatic — use def instead)
square = lambda x: x ** 2

# Good style
def square(x):
    return x ** 2
```

---

### `map(function, iterable)`

Think of it as a **factory conveyor belt** — every item goes in, gets transformed, comes out the other side. Returns a **lazy iterator** — wrap with `list()` to materialize it.

```python
nums = [1, 2, 3, 4]

list(map(lambda x: x ** 2, nums))
# [1, 4, 9, 16]

# Built-in functions work directly — no lambda needed
list(map(str, nums))              # ['1', '2', '3', '4']
list(map(int, ['1', '2', '3']))   # [1, 2, 3]

# map over multiple iterables in parallel
list(map(lambda x, y: x + y, [1, 2], [10, 20]))
# [11, 22]
```

##### Real-world examples
```python
names = [' alice ', ' bob ']
list(map(str.strip, names))        # ['alice', 'bob']

prices = ['1.5', '2.3', '4.0']
list(map(float, prices))           # [1.5, 2.3, 4.0]

words = ['hello', 'world']
list(map(str.upper, words))        # ['HELLO', 'WORLD']
```

##### map vs list comprehension
```python
list(map(lambda x: x**2, nums))    # functional style
[x**2 for x in nums]               # Pythonic style — generally preferred
```
Both give the same result. List comprehensions are more readable per PEP 8 convention. `map` shines when paired with a built-in function directly (`map(str, nums)`).

---

### `filter(function, iterable)`

Think of it as a **bouncer at a club** — only items that pass the test get through. Function must return `True`/`False`. Also returns a lazy iterator.

```python
nums = [1, 2, 3, 4, 5, 6]

list(filter(lambda x: x % 2 == 0, nums))
# [2, 4, 6]
```

##### The `filter(None, ...)` trick
Removes all falsy values (`None`, `0`, `''`, `[]`, `False`) in one shot:
```python
data = [0, 1, '', 'a', None, [], [1], False, True]
list(filter(None, data))
# [1, 'a', [1], True]
```

##### Real-world examples
```python
users = [
    {'name': 'Alice', 'active': True},
    {'name': 'Bob', 'active': False},
    {'name': 'Eve', 'active': True}
]
active = list(filter(lambda u: u['active'], users))
# Alice and Eve only

scores = [None, 85, None, 92]
valid = list(filter(None, scores))
# [85, 92]
```

##### filter vs list comprehension
```python
list(filter(lambda x: x > 3, nums))     # functional style
[x for x in nums if x > 3]              # Pythonic style — generally preferred
```
`filter(None, lst)` is the one case where `filter` genuinely reads cleaner than the comprehension equivalent.

---

### `reduce(function, iterable, initializer=None)`

Think of it as a **snowball rolling downhill** — grabs the first two items, combines them, rolls that result into the next item, and so on until one value remains. **Not a built-in** — must import from `functools`.

```python
from functools import reduce

nums = [1, 2, 3, 4]

reduce(lambda acc, x: acc + x, nums)
# Step: (1+2)=3 -> (3+3)=6 -> (6+4)=10
# Result: 10

reduce(lambda acc, x: acc * x, nums)
# Step: (1*2)=2 -> (2*3)=6 -> (6*4)=24
# Result: 24
```

##### With an initializer (safer — also handles empty lists)
```python
reduce(lambda acc, x: acc + x, nums, 100)
# 100 -> 101 -> 103 -> 106 -> 110

reduce(lambda a, b: a + b, [], 0)   # 0 — safe on empty list
reduce(lambda a, b: a + b, [])      # TypeError: empty sequence, no initial value!
```

##### Real-world examples
```python
# Flatten a list of lists
nested = [[1,2], [3,4], [5,6]]
reduce(lambda a, b: a + b, nested)
# [1, 2, 3, 4, 5, 6]

# Max without max()
reduce(lambda a, b: a if a > b else b, [3, 1, 4, 1, 5, 9])
# 9

# Concatenate strings
words = ['Hello', ' ', 'World']
reduce(lambda a, b: a + b, words)
# 'Hello World'

# Build a dict from pairs
pairs = [('a', 1), ('b', 2)]
reduce(lambda acc, p: {**acc, p[0]: p[1]}, pairs, {})
# {'a': 1, 'b': 2}
```

`reduce` has **no comprehension equivalent** — it's the one of the three you'll actually reach for in real code, since `map`/`filter` are usually replaced by comprehensions in idiomatic Python.

---

#### Recognizing which one you need (interview signal words)

| Signal phrase in the problem | Use |
|---|---|
| "transform every element" / "apply X to each item" | `map` |
| "keep only elements that..." / "remove elements where..." | `filter` |
| "combine all elements into one" / "total", "product", "flatten" | `reduce` |
| "square each, then keep evens, then sum" (chained) | `filter` → `map` → `reduce` |
| "sort by a specific field" | `lambda` with `key=` |

---

#### The chain pattern — read inside-out

```python
from functools import reduce

nums = [1, 2, 3, 4, 5, 6]

result = reduce(
    lambda acc, x: acc + x,                    # 3. sum them
    map(lambda x: x**2,                         # 2. square each
        filter(lambda x: x % 2 == 0, nums))      # 1. keep evens first
)
# evens: [2,4,6] -> squared: [4,16,36] -> sum: 56
print(result)  # 56
```

---

#### Common pitfalls (the ones that actually bite people)

##### Pitfall 1 — `map`/`filter` are lazy; printing them shows garbage
```python
result = map(lambda x: x*2, [1,2,3])
print(result)          # <map object at 0x...> — NOT the values!
print(list(result))    # [2, 4, 6] — need list() to materialize
```

##### Pitfall 2 — iterators get exhausted after one use
```python
m = map(lambda x: x*2, [1,2,3])
list(m)   # [2, 4, 6]
list(m)   # [] — already consumed! Iterators don't reset.
```

##### Pitfall 3 — forgetting to import `reduce`
```python
reduce(...)                     # NameError if you forgot this:
from functools import reduce    # required every time — not built-in like map/filter
```

##### Pitfall 4 — `reduce` crashes on an empty list without an initializer
```python
reduce(lambda a, b: a+b, [])       # TypeError: empty sequence
reduce(lambda a, b: a+b, [], 0)    # 0 — always provide an initial value defensively
```

##### Pitfall 5 — using `map`/`filter` when a comprehension is clearer (style, not a bug)
```python
# Works, but not idiomatic Python
list(map(lambda x: x**2, filter(lambda x: x%2==0, nums)))

# Preferred
[x**2 for x in nums if x % 2 == 0]
```

##### Pitfall 6 — assigning a lambda to a name instead of using `def`
```python
# Not idiomatic
square = lambda x: x ** 2

# Idiomatic
def square(x):
    return x ** 2
```
---

### Python `range`, `enumerate`, `zip` — Complete Interview Notes

---
| Function | Does | Returns |
|---|---|---|
| `range(start, stop, step)` | generates a sequence of integers | lazy range object |
| `enumerate(iterable, start=0)` | adds an index counter to any iterable | lazy iterator of `(index, item)` tuples |
| `zip(*iterables)` | pairs up elements from multiple iterables by position | lazy iterator of tuples |

---

### `range(start, stop, step)`

Think of it as a **number tap** — produces integers on demand without storing them all at once. Stop is **always excluded**.

```python
range(5)          # 0 1 2 3 4          (stop only)
range(2, 6)       # 2 3 4 5            (start + stop)
range(0, 10, 2)   # 0 2 4 6 8          (with step — even numbers)
range(10, 0, -1)  # 10 9 8 ... 1       (negative step — countdown)
range(5, 2)       # empty              (start >= stop with positive step)
```

##### Common uses

```python
# loop N times
for i in range(5):
    print(i)

# repeat something N times (index not needed)
for _ in range(3):
    do_something()

# index-based loop (when you need to mutate the list)
for i in range(len(lst)):
    lst[i] = lst[i] * 2

# countdown
for i in range(10, 0, -1):
    print(i)

# convert to a list
list(range(5))    # [0, 1, 2, 3, 4]
```

##### Key facts for interviews

- `range` is **lazy** — it doesn't store all values in memory. `range(10**9)` is fine.
- It produces **integers only** — not floats.
- It is a `range` object, not a `list`. Wrap with `list()` to materialise.
- Supports `len()`, indexing, and `in` checks without iterating: `5 in range(10)` is `O(1)`.

---

### `enumerate(iterable, start=0)`

Think of it as a **ticket machine** — hands each item a number as it exits the conveyor belt. Always unpack two values: `(index, item)`.

```python
fruits = ['apple', 'banana', 'cherry']

for i, fruit in enumerate(fruits):
    print(i, fruit)
# 0 apple
# 1 banana
# 2 cherry

# start from 1
for i, fruit in enumerate(fruits, start=1):
    print(i, fruit)
# 1 apple
# 2 banana
# 3 cherry
```

##### Common uses

```python
# numbered output
for i, item in enumerate(items, 1):
    print(f"{i}. {item}")

# modify a list in place
for i, val in enumerate(lst):
    lst[i] = val.upper()

# find index of a matching item
for i, val in enumerate(data):
    if val == target:
        print(f"found at index {i}")

# works on any iterable — strings, tuples, files
for i, ch in enumerate("hello"):
    print(i, ch)
```

##### `enumerate` vs `range(len(...))`

```python
# bad — old style, verbose
i = 0
for item in items:
    print(i, item)
    i += 1

# bad — works but not idiomatic
for i in range(len(items)):
    print(i, items[i])

# good — always prefer enumerate when you need both index and value
for i, item in enumerate(items):
    print(i, item)
```

**When to still use `range(len(...))`:**
- Mutating the list in place and needing the index for `lst[i] = ...`
- Comparing adjacent elements: `nums[i]` vs `nums[i+1]`
- Doing index arithmetic: `i-1`, `i+2`, etc.
- Looping two separate lists by the same index (though `zip` is usually better)

---

### `zip(*iterables)`

Think of it as a **jacket zipper** — locks corresponding teeth from two sides together. Stops at the **shortest** iterable — extra elements are silently dropped.

```python
names  = ['Alice', 'Bob', 'Charlie']
scores = [95, 87, 73]

for name, score in zip(names, scores):
    print(name, score)
# Alice 95
# Bob 87
# Charlie 73

# zip 3 iterables at once
a, b, c = [1, 2], ['x', 'y'], [True, False]
list(zip(a, b, c))
# [(1, 'x', True), (2, 'y', False)]
```

##### Stops at the shortest — the silent drop pitfall

```python
a = [1, 2, 3, 4]
b = ['x', 'y']

list(zip(a, b))
# [(1, 'x'), (2, 'y')]   <- 3 and 4 silently dropped!
```

##### `zip_longest` — when you want to keep all elements

```python
from itertools import zip_longest

a = [1, 2, 3, 4]
b = ['x', 'y']

list(zip_longest(a, b, fillvalue='?'))
# [(1, 'x'), (2, 'y'), (3, '?'), (4, '?')]
```

##### Starting mid-way — `zip` has no `start=` param

```python
a = [10, 20, 30, 40, 50]
b = ['x', 'y', 'z', 'w', 'v']

# slice before zipping
for x, y in zip(a[2:], b[2:]):
    print(x, y)
# 30 z   40 w   50 v

# memory-efficient version (no copy)
from itertools import islice
for x, y in zip(islice(a, 2, None), islice(b, 2, None)):
    print(x, y)
```

##### Common uses

```python
# iterate two lists together
for name, score in zip(names, scores):
    print(f"{name}: {score}")

# build a dict from two lists
keys   = ['a', 'b', 'c']
values = [1, 2, 3]
d = dict(zip(keys, values))   # {'a': 1, 'b': 2, 'c': 3}

# transpose a matrix (rows to columns)
matrix = [[1, 2, 3], [4, 5, 6]]
transposed = list(zip(*matrix))   # [(1, 4), (2, 5), (3, 6)]

# unzip (reverse of zip)
pairs = [(1, 'a'), (2, 'b'), (3, 'c')]
nums, letters = zip(*pairs)       # (1,2,3)  ('a','b','c')
```

---

#### The power combo — `enumerate` + `zip`

Interviewers love seeing this — gives you position + paired values in one loop.

```python
names  = ['Alice', 'Bob', 'Charlie']
scores = [95, 87, 73]

for i, (name, score) in enumerate(zip(names, scores), 1):
    print(f"{i}. {name} -> {score}")
# 1. Alice -> 95
# 2. Bob -> 87
# 3. Charlie -> 73
```

---

#### When to use which — decision guide

| I need to... | Use |
|---|---|
| Loop N times | `range(N)` |
| Loop from 5 to 20 in steps of 3 | `range(5, 21, 3)` |
| Loop a list and know position | `enumerate(lst)` |
| Loop two lists element by element | `zip(a, b)` |
| Loop two lists + know position | `enumerate(zip(a, b))` |
| Combine lists of unequal length | `zip_longest(a, b, fillvalue=...)` |
| Start zip mid-way | `zip(a[i:], b[i:])` |
| Mutate list in place | `range(len(lst))` |
| Compare adjacent elements | `range(len(lst) - 1)` |

---

#### Common pitfalls

##### Pitfall 1 — `zip` silently drops extra elements
```python
a = [1, 2, 3, 4]
b = ['x', 'y']
list(zip(a, b))   # [(1,'x'),(2,'y')] — 3 and 4 are gone with no warning
# fix: use zip_longest if unequal lengths are possible
```

##### Pitfall 2 — `range` stop is excluded
```python
range(1, 5)    # 1 2 3 4  — NOT 5
range(5)       # 0 1 2 3 4 — NOT 5
# common bug: range(len(lst)) is correct, range(len(lst)+1) is wrong
```

##### Pitfall 3 — `range` is not a list
```python
r = range(5)
print(r)         # range(0, 5) — not the values!
print(list(r))   # [0, 1, 2, 3, 4]
```

##### Pitfall 4 — forgetting to unpack `enumerate`
```python
for x in enumerate(items):
    print(x)          # prints (0, 'item') — the whole tuple

for i, x in enumerate(items):
    print(i, x)       # correct — unpacked
```

##### Pitfall 5 — using `range(len(...))` when `enumerate` is cleaner
```python
# not idiomatic
for i in range(len(nums)):
    print(i, nums[i])

# idiomatic
for i, val in enumerate(nums):
    print(i, val)
```

##### Pitfall 6 — `zip` with a single iterable returns 1-tuples
```python
list(zip([1, 2, 3]))
# [(1,), (2,), (3,)]  — each element wrapped in a tuple
# this is rarely what you want — check you're passing two iterables
```

---
